# Clasificación de diabetes a partir de indicadores clínicos y demográficos
**Minería de Datos — Actividad 1, tercer corte**

**Integrantes:** José Rivas, Sebastián Muñoz y Camilo Camargo  
**Tipo de problema:** clasificación supervisada.  
**Variable objetivo:** `Outcome` (0 = clase negativa, 1 = clase positiva).  
**Pregunta del proyecto:** ¿Qué tan bien pueden la regresión logística y un árbol de decisión clasificar `Outcome` a partir de ocho indicadores clínicos y demográficos?

Este notebook actualiza el flujo del segundo corte para revisar la calidad de los datos, comparar los modelos, comprobar el desempeño con validación cruzada, guardar el pipeline completo y verificar que el modelo serializado produzca la misma predicción.

**Aviso:** este es un ejercicio académico. El conjunto de datos no representa a toda la población y el modelo no diagnostica ni sustituye la valoración de profesionales de la salud.

## 0. Preparación del entorno y configuración

Ejecuta las celdas en orden. El notebook intenta leer `diabetes.csv` desde la misma carpeta. Si no lo encuentra, intenta descargar la versión pública del conjunto Pima Indians Diabetes. Si la descarga no está disponible, puedes colocar el CSV del segundo corte en la carpeta del notebook con el nombre `diabetes.csv` y volver a ejecutar esta celda.

La fuente de referencia del proyecto continúa siendo el conjunto indicado en el segundo informe de avance: [Kaggle — Diabetes Dataset](https://www.kaggle.com/datasets/amineipad/diabetes-dataset). Para facilitar la reproducción también se deja un espejo público con el formato estándar Pima.

In [10]:
from pathlib import Path
import json
import warnings
import urllib.request

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
DATA_FILE = Path("diabetes.csv")
MODEL_FILE = Path("pipeline_diabetes.joblib")
METRICS_FILE = Path("metricas_modelo.json")
KNOWN_CASE_FILE = Path("caso_prueba_conocido.json")

DATA_URL = (
    "https://raw.githubusercontent.com/npradaschnor/"
    "Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
)

FEATURES = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]
TARGET = "Outcome"
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Entorno preparado.")
print("Carpeta de trabajo:", Path.cwd())

ModuleNotFoundError: No module named 'joblib'

## 1. Carga y validación inicial del conjunto de datos

Se comprueba que existan las ocho variables predictoras y la variable objetivo. También se validan los tipos de datos y que `Outcome` contenga únicamente las clases 0 y 1. No se generan registros artificiales: los resultados se calculan con el CSV real que se cargue.

In [ ]:
def load_dataset():
    if DATA_FILE.exists():
        data = pd.read_csv(DATA_FILE)
        print(f"Archivo local cargado: {DATA_FILE.resolve()}")
    else:
        try:
            with urllib.request.urlopen(DATA_URL, timeout=20) as response:
                from io import BytesIO
                data = pd.read_csv(BytesIO(response.read()))
            print("Dataset descargado desde el espejo público.")
        except Exception as exc:
            raise FileNotFoundError(
                "No fue posible encontrar ni descargar diabetes.csv. "
                "Coloca el CSV real del proyecto en la misma carpeta con ese nombre "
                "y vuelve a ejecutar. No se han creado datos ficticios."
            ) from exc

    missing_columns = [c for c in FEATURES + [TARGET] if c not in data.columns]
    if missing_columns:
        raise ValueError(
            f"El CSV no contiene las columnas esperadas: {missing_columns}. "
            "Verifica que corresponda al dataset Pima Indians del proyecto."
        )

    data = data[FEATURES + [TARGET]].copy()
    for column in FEATURES + [TARGET]:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    data = data.dropna(subset=[TARGET])
    if not set(data[TARGET].unique()).issubset({0, 1}):
        raise ValueError("La columna Outcome debe contener únicamente 0 y 1.")

    data[TARGET] = data[TARGET].astype(int)
    if data.empty:
        raise ValueError("El conjunto de datos está vacío después de la validación.")
    return data

df = load_dataset()

print("Dimensiones (filas, columnas):", df.shape)
print("\nPrimeros registros:")
display(df.head())
print("\nTipos de datos:")
display(df.dtypes.to_frame("Tipo"))
print("\nResumen estadístico:")
display(df.describe().T)
print("\nValores faltantes por columna:")
display(df.isna().sum().to_frame("Valores faltantes"))
print("\nFilas duplicadas:", int(df.duplicated().sum()))
print("\nDistribución de Outcome:")
display(df[TARGET].value_counts().sort_index().to_frame("Cantidad"))
print("\nProporción de Outcome:")
display((df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2).to_frame("Porcentaje"))

## 2. Análisis exploratorio de datos (EDA)

Se revisa la distribución de la variable objetivo, la glucosa por clase, la correlación entre variables y los ceros observados en mediciones donde un cero puede representar un dato no registrado. Estas gráficas describen el conjunto, pero no demuestran causalidad.

In [ ]:
# Distribución de la variable objetivo
counts = df[TARGET].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Distribución de la variable Outcome")
plt.xlabel("Outcome")
plt.ylabel("Número de registros")
plt.tight_layout()
plt.show()

# Distribución de glucosa separada por clase
plt.figure(figsize=(8, 4))
for outcome in sorted(df[TARGET].unique()):
    values = df.loc[df[TARGET] == outcome, "Glucose"].dropna()
    plt.hist(values, bins=25, alpha=0.55, label=f"Outcome {outcome}")
plt.title("Distribución de glucosa según Outcome")
plt.xlabel("Glucosa")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()

# Matriz de correlación
corr = df[FEATURES + [TARGET]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=75, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Matriz de correlación")
fig.colorbar(im, ax=ax, label="Correlación")
plt.tight_layout()
plt.show()

# Posibles valores no registrados codificados como cero
zero_counts = (df[ZERO_AS_MISSING] == 0).sum().sort_values(ascending=False)
display(zero_counts.to_frame("Ceros observados"))

## 3. Preparación de los datos y control de calidad

En `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` y `BMI`, los ceros se tratan como posibles mediciones no registradas, siguiendo el criterio metodológico del segundo avance. La imputación por mediana se incorpora dentro de cada pipeline; de esta manera, la mediana se aprende solamente con los datos usados para entrenar cada partición y se reduce el riesgo de fuga de información.

Se conserva `Outcome` como objetivo y las ocho variables originales como predictoras.

In [ ]:
def prepare_features(X):
    X = X.copy()
    for column in ZERO_AS_MISSING:
        X[column] = X[column].replace(0, np.nan)
    return X

X = prepare_features(df[FEATURES])
y = df[TARGET].copy()

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)
print("\nValores faltantes después de revisar ceros:")
display(X.isna().sum().to_frame("Faltantes"))
print("\nBalance de clases:")
display(y.value_counts().sort_index().to_frame("Cantidad"))

# División estratificada: conserva aproximadamente la proporción de las clases.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Registros de entrenamiento:", len(X_train))
print("Registros de prueba:", len(X_test))
print("Proporción de clase positiva en entrenamiento:", round(y_train.mean(), 3))
print("Proporción de clase positiva en prueba:", round(y_test.mean(), 3))

## 4. Construcción de los modelos

Se comparan los dos algoritmos conservados del proyecto: regresión logística y árbol de decisión. Ambos pipelines incluyen imputación por mediana. La regresión logística también incorpora escalamiento, mientras que el árbol de decisión no requiere esa transformación.

In [ ]:
def build_models():
    logistic_preprocessing = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), FEATURES)
        ],
        remainder="drop"
    )

    tree_preprocessing = ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median"), FEATURES)
        ],
        remainder="drop"
    )

    logistic_pipeline = Pipeline(steps=[
        ("preprocessing", logistic_preprocessing),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    tree_pipeline = Pipeline(steps=[
        ("preprocessing", tree_preprocessing),
        ("model", DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=8,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    return {
        "Regresión logística": logistic_pipeline,
        "Árbol de decisión": tree_pipeline
    }

models = build_models()
print("Modelos construidos:", ", ".join(models.keys()))

## 5. Entrenamiento y validación cruzada

La evaluación utiliza una partición de entrenamiento y prueba estratificada y validación cruzada estratificada de cinco particiones sobre los datos de entrenamiento. Se reportan accuracy, precision, recall y F1 para la clase positiva, además del promedio y desviación estándar del F1 en validación cruzada.

La selección no se basa únicamente en accuracy: se prioriza recall de la clase positiva y se utilizan F1 y precision como criterios de desempate.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

fitted_models = {}
evaluation = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    cv_result = cross_validate(
        pipeline, X_train, y_train,
        cv=cv, scoring=scoring, return_train_score=False
    )

    evaluation[name] = {
        "accuracy": float(accuracy_score(y_test, test_pred)),
        "precision_1": float(precision_score(y_test, test_pred, zero_division=0)),
        "recall_1": float(recall_score(y_test, test_pred, zero_division=0)),
        "f1_1": float(f1_score(y_test, test_pred, zero_division=0)),
        "train_accuracy": float(accuracy_score(y_train, train_pred)),
        "cv_accuracy_mean": float(np.mean(cv_result["test_accuracy"])),
        "cv_precision_mean": float(np.mean(cv_result["test_precision"])),
        "cv_recall_mean": float(np.mean(cv_result["test_recall"])),
        "cv_f1_mean": float(np.mean(cv_result["test_f1"])),
        "cv_f1_std": float(np.std(cv_result["test_f1"], ddof=1)),
        "confusion_matrix": confusion_matrix(y_test, test_pred, labels=[0, 1]).tolist(),
        "classification_report": classification_report(
            y_test, test_pred, labels=[0, 1],
            output_dict=True, zero_division=0
        )
    }
    fitted_models[name] = pipeline

comparison = pd.DataFrame(evaluation).T[
    ["accuracy", "precision_1", "recall_1", "f1_1",
     "train_accuracy", "cv_f1_mean", "cv_f1_std"]
]
display(comparison.round(4))

## 6. Comparación de errores y matrices de confusión

La matriz de confusión permite distinguir verdaderos positivos, verdaderos negativos, falsos positivos y falsos negativos. En este problema se debe prestar especial atención a los falsos negativos y al recall de la clase 1.

In [ ]:
for name, pipeline in fitted_models.items():
    predictions = pipeline.predict(X_test)
    print("\n", "=" * 60)
    print(name)
    print(classification_report(y_test, predictions, labels=[0, 1], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions, labels=[0, 1], values_format="d"
    )
    plt.title(f"Matriz de confusión — {name}")
    plt.tight_layout()
    plt.show()

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()
    print(f"TN={tn} | FP={fp} | FN={fn} | TP={tp}")

## 7. Selección del modelo y revisión de sobreajuste

Se elige el modelo con mayor recall de la clase positiva. Si hay empate, se comparan F1 y precision. También se observa la diferencia entre accuracy de entrenamiento y prueba como una señal descriptiva de posible sobreajuste; esta diferencia no debe interpretarse de forma aislada.

In [ ]:
selected_name = max(
    evaluation,
    key=lambda name: (
        evaluation[name]["recall_1"],
        evaluation[name]["f1_1"],
        evaluation[name]["precision_1"]
    )
)
selected_pipeline = fitted_models[selected_name]
selected_result = evaluation[selected_name]

print("Modelo seleccionado:", selected_name)
print("Criterio: recall de clase 1; desempate por F1 y precision.")
print("Recall clase 1:", round(selected_result["recall_1"], 4))
print("F1 clase 1:", round(selected_result["f1_1"], 4))
print("Precision clase 1:", round(selected_result["precision_1"], 4))
print("Accuracy entrenamiento:", round(selected_result["train_accuracy"], 4))
print("Accuracy prueba:", round(selected_result["accuracy"], 4))
print("Diferencia train-test:", round(
    selected_result["train_accuracy"] - selected_result["accuracy"], 4
))

if selected_result["accuracy"] < 0.70:
    print("ALERTA: accuracy inferior a 0.70. Revisar el pipeline y la ficha técnica de calidad.")
else:
    print("Accuracy igual o superior a 0.70; aun así deben revisarse recall, falsos negativos y estabilidad de CV.")

## 8. Ajustes realizados respecto al flujo anterior

Para esta actualización se deja explícito el tratamiento de ceros potencialmente no registrados, se mantiene la imputación dentro de los pipelines para reducir fuga de información, se compara regresión logística con árbol de decisión y se incorporan validación cruzada, métricas por clase, revisión de falsos negativos y una comprobación del modelo serializado.

**Comparación con las métricas numéricas del segundo corte:** el material disponible para preparar esta versión contiene el notebook y el informe de avance, pero no un archivo de métricas ejecutadas del segundo corte. Por esa razón no se inventan cifras anteriores ni una mejora porcentual. La tabla del apartado 5 constituye la comparación reproducible de los dos modelos con el conjunto cargado en esta ejecución. Si el grupo conserva el JSON o la tabla de métricas originales, debe agregarse para realizar una comparación antes/después.

## 9. Entrenamiento final y exportación del pipeline

Una vez seleccionado el algoritmo con la evaluación de entrenamiento/prueba, se ajusta el pipeline seleccionado con todos los registros disponibles y se guarda con `joblib`. El archivo incluye el preprocesamiento y el clasificador; Streamlit debe cargar este artefacto y no volver a entrenar el modelo en cada ejecución.

También se exportan las métricas obtenidas y un caso conocido para comprobar la consistencia entre notebook y aplicación.

In [ ]:
# Reentrenar el pipeline seleccionado con todo el conjunto disponible.
final_pipeline = build_models()[selected_name]
final_pipeline.fit(X, y)

joblib.dump(final_pipeline, MODEL_FILE)

# Caso de prueba tomado de la partición de prueba, no un registro inventado.
known_case = X_test.iloc[[0]].copy()
known_prediction = int(final_pipeline.predict(known_case)[0])

known_case_values = {}
for column, value in known_case.iloc[0].items():
    known_case_values[column] = None if pd.isna(value) else float(value)

known_payload = {
    "source": "Primer registro de X_test de esta ejecución",
    "features": known_case_values,
    "expected_prediction": known_prediction,
    "selected_model": selected_name
}
with open(KNOWN_CASE_FILE, "w", encoding="utf-8") as file:
    json.dump(known_payload, file, ensure_ascii=False, indent=2)

metrics_payload = {
    "selected_model": selected_name,
    "selection_rule": "Mayor recall de clase 1; desempate por F1 y precision",
    "dataset_rows": int(df.shape[0]),
    "dataset_columns": int(df.shape[1]),
    "test_size": 0.20,
    "random_state": RANDOM_STATE,
    "cross_validation_folds": 5,
    "models": evaluation
}
with open(METRICS_FILE, "w", encoding="utf-8") as file:
    json.dump(metrics_payload, file, ensure_ascii=False, indent=2)

print("Archivos generados:")
print("-", MODEL_FILE.resolve())
print("-", METRICS_FILE.resolve())
print("-", KNOWN_CASE_FILE.resolve())

## 10. Prueba del artefacto serializado y predicción de un caso conocido

Se vuelve a cargar el pipeline guardado y se comprueba que la predicción del caso conocido sea idéntica a la obtenida antes de guardar. Este caso puede utilizarse para verificar que Streamlit está cargando el mismo modelo.

In [ ]:
loaded_pipeline = joblib.load(MODEL_FILE)
prediction_before_save = int(final_pipeline.predict(known_case)[0])
prediction_after_load = int(loaded_pipeline.predict(known_case)[0])

print("Predicción antes de guardar:", prediction_before_save)
print("Predicción después de cargar:", prediction_after_load)
assert prediction_before_save == prediction_after_load, (
    "Error: las predicciones cambiaron después de cargar el pipeline."
)
assert prediction_after_load == known_prediction
print("PRUEBA SUPERADA: el pipeline serializado conserva la predicción.")

if hasattr(loaded_pipeline, "predict_proba"):
    probability = loaded_pipeline.predict_proba(known_case)[0]
    print("Probabilidad clase 0:", round(float(probability[0]), 4))
    print("Probabilidad clase 1:", round(float(probability[1]), 4))

## 11. Validación de una entrada y prueba de rangos

La aplicación debe validar tipos y rangos antes de predecir. La siguiente función sirve como referencia para validar una fila de entrada. Los límites son controles de plausibilidad para el formulario, no criterios médicos ni rangos diagnósticos.

In [ ]:
FORM_RANGES = {
    "Pregnancies": (0, 20),
    "Glucose": (1, 300),
    "BloodPressure": (1, 200),
    "SkinThickness": (0, 100),
    "Insulin": (0, 900),
    "BMI": (1, 80),
    "DiabetesPedigreeFunction": (0, 3),
    "Age": (1, 120)
}

def validate_input(record):
    errors = []
    for column in FEATURES:
        if column not in record:
            errors.append(f"Falta la variable {column}.")
            continue
        try:
            value = float(record[column])
        except (TypeError, ValueError):
            errors.append(f"{column} debe ser numérica.")
            continue
        lower, upper = FORM_RANGES[column]
        if not np.isfinite(value) or value < lower or value > upper:
            errors.append(f"{column} debe estar entre {lower} y {upper}.")
    return errors

valid_example = {column: float(known_case.iloc[0][column])
                 if pd.notna(known_case.iloc[0][column]) else 1.0
                 for column in FEATURES}
print("Errores de ejemplo válido:", validate_input(valid_example))

invalid_example = valid_example.copy()
invalid_example["Age"] = -5
print("Errores de ejemplo inválido:", validate_input(invalid_example))

## 12. Resultados, interpretación y conclusión

La conclusión se construye a partir de los valores calculados al ejecutar el notebook. No se escriben porcentajes fijos para evitar reportar resultados que no correspondan al CSV utilizado. Revisa especialmente recall de la clase positiva, falsos negativos, F1 y la variación de la validación cruzada.

El conjunto es histórico y específico; los ceros tratados como faltantes pueden representar mediciones no registradas y la imputación no recupera los valores reales. El desempeño en una partición no garantiza generalización a otros grupos de población. El modelo es un prototipo académico y no debe utilizarse para diagnóstico ni decisiones clínicas.

In [ ]:
print(f"El modelo seleccionado fue: {selected_name}.")
print(f"Accuracy en prueba: {selected_result['accuracy']:.3f}.")
print(f"Precision clase 1: {selected_result['precision_1']:.3f}.")
print(f"Recall clase 1: {selected_result['recall_1']:.3f}.")
print(f"F1 clase 1: {selected_result['f1_1']:.3f}.")
print(f"F1 promedio en validación cruzada: {selected_result['cv_f1_mean']:.3f} "
      f"(desviación estándar: {selected_result['cv_f1_std']:.3f}).")
print("Interpreta estos resultados junto con la matriz de confusión y los falsos negativos.")

## 13. Evaluación del uso de inteligencia artificial

- **Herramienta utilizada:** ChatGPT.
- **Propósito:** apoyar la organización del notebook, revisar el flujo de preparación de datos, proponer comprobaciones de calidad y documentar la integración del pipeline con Streamlit.
- **Prompt principal:** “Ayúdame a actualizar un notebook reproducible para el proyecto de clasificación de diabetes con Pima Indians. Conserva regresión logística y árbol de decisión; incluye EDA, tratamiento justificado de ceros, división estratificada, validación cruzada, métricas por clase, matrices de confusión, selección basada en recall, exportación con joblib, prueba de consistencia y limitaciones”.
- **Cómo se verifica la respuesta:** ejecutar las celdas en orden con el CSV real, revisar nombres y dimensiones, inspeccionar las métricas y matrices de confusión, comprobar que el archivo `pipeline_diabetes.joblib` se genere y verificar que la predicción antes y después de cargarlo sea idéntica.
- **Problemas encontrados y ajustes:** no se dispone en los archivos consultados de una tabla de métricas ejecutadas del segundo corte; por eso no se inventó una comparación numérica antes/después. Se mantuvo la estructura del proyecto y se documentaron los cambios metodológicos para poder verificarlos con los datos reales.
- **Responsabilidad del grupo:** José Rivas, Sebastián Muñoz y Camilo Camargo deben ejecutar el notebook, revisar los resultados, comprender el código y comprobar la aplicación publicada antes de entregar.

La IA se utilizó como apoyo y no como sustituto de la revisión del conjunto de datos, la interpretación de resultados ni las decisiones del grupo.

## 14. Lista de comprobación antes de entregar

- [ ] El CSV utilizado corresponde al conjunto de datos del proyecto y tiene las columnas esperadas.
- [ ] Todas las celdas se ejecutaron sin errores.
- [ ] Se revisaron las métricas, el recall de clase 1 y los falsos negativos.
- [ ] Se generaron `pipeline_diabetes.joblib`, `metricas_modelo.json` y `caso_prueba_conocido.json`.
- [ ] El pipeline cargado produce la misma predicción que antes de serializarlo.
- [ ] El repositorio incluye `app.py`, `requirements.txt`, `README.md`, el pipeline serializado, este notebook y los archivos auxiliares necesarios.
- [ ] La aplicación Streamlit muestra los nombres de los tres integrantes y funciona con el pipeline generado.
- [ ] Se comprobó la URL pública de Streamlit y el repositorio de GitHub.

**Importante:** este notebook está estructurado para generar resultados reales al ejecutarse con el CSV. No incluye métricas inventadas ni resultados de ejecución que no hayan sido calculados.